In [1]:
from JointTemporalModel import JointTemporalModel
import torch, math
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from Utils import collator, TemporalDataset
from torch.utils.data import DataLoader

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
label2id_ner = {"B-DATE":0, "B-DURATION":1, "B-EVENT": 2, "B-TIME": 3, "I-DATE":4, "I-DURATION":5, "I-EVENT":6, "I-TIME":7, "O":8}
id2label_ner = {0: 'B-DATE', 1: 'B-DURATION', 2: 'B-EVENT', 3: 'B-TIME', 4: 'I-DATE', 5: 'I-DURATION', 6: 'I-EVENT', 7: 'I-TIME', 8: 'O'}
label2id_ee = {"AFTER": 0, "BEFORE": 1, "CONTAINS": 2, "DURING":3, "EQUALS":4, "IDENTITY":5, "OVERLAPS":6}
id2label_ee = {0: 'AFTER', 1: 'BEFORE', 2: 'CONTAINS', 3: 'DURING', 4: 'EQUALS', 5: 'IDENTITY', 6: 'OVERLAPS'}
cleandata_path = "D:\\GeoTKG\\cleandata\\tie\\"
def collate_fn(examples):
    return collator(examples, label2id_ner=label2id_ner, label2id_ee=label2id_ee)
train = TemporalDataset(cleandata_path + "train.json")
eval = TemporalDataset(cleandata_path + "eval.json")
train_loader = DataLoader(train, batch_size=8, shuffle=True, collate_fn=collate_fn)
eval_loader = DataLoader(eval, batch_size=8, shuffle=False, collate_fn=collate_fn)

In [3]:
NUM_EPOCHS = 15
GRAD_ACCUM = 1
ENC_LR = 5e-5
NONENC_LR = 1e-3
WARMUP_EPOCHS = 2
LOGGING_STEPS = 200
BASE_ENC_MODEL = "roberta-base"

In [4]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model = JointTemporalModel(base=BASE_ENC_MODEL, num_ner=len(label2id_ner), ee_labels=len(label2id_ee), heads=4).to(device)

#for p in model.enc.parameters(): p.requires_grad = False

optimizer = AdamW([
        {"params": [p for n,p in model.named_parameters() if n.startswith("enc.")], "lr": ENC_LR},
        {"params": [p for n,p in model.named_parameters() if not n.startswith("enc.")], "lr": NONENC_LR},
    ], weight_decay=0.01)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
num_train_steps = steps_per_epoch * NUM_EPOCHS
num_warmup_steps = steps_per_epoch * WARMUP_EPOCHS

sched = get_linear_schedule_with_warmup(optimizer, int(0.05*num_train_steps), num_train_steps)

scaler = torch.amp.GradScaler(enabled=(device.type=='cuda'))

def make_optim(unfrozen: bool):
    groups = []
    if unfrozen:
        groups.append({"params": model.enc.parameters(), "lr": ENC_LR})
    else:
        # keep enc group empty or skip entirely; either is fine
        pass
    nonenc = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
    groups.append({"params": nonenc, "lr": NONENC_LR})
    return AdamW(groups, weight_decay=0.01)

Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.dense.weight', 'lm_head.bias', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
global_step = 0
history = {"loss": [], "ner_loss":[], "ca_loss":[], "ee_loss":[], "ner_f1": [], "ptr_acc": [], "ee_f1": []}
for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}

        ctx = (torch.autocast(device_type='cuda', dtype=torch.float16))
        with ctx:
            # Ensure your model.forward signature matches these keys
            out = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                ev_starts=batch["ev_starts"], ev_ends=batch["ev_ends"], ev_mask=batch["ev_mask"],
                ti_starts=batch["ti_starts"], ti_ends=batch["ti_ends"], ti_mask=batch["ti_mask"],
                ner_gold_labels=batch["ner_labels"],
                ev_ti_gold=batch["ev_ti_gold"],
                ee_rel_gold=batch["ee_triples"],
                ee_mask=batch["ee_mask"],
            )
            loss = out["loss"] / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            sched.step()
            global_step += 1
            
        if global_step % LOGGING_STEPS == 0:
            print(f"LOSS: {loss.item():.4f}, STEP: {global_step}")
            history["loss"].append(loss.item())
            history["ner_loss"].append(out["ner_loss"].item())
            history["ca_loss"].append(out["ca_loss"].item())
            history["ee_loss"].append(out["ee_loss"].item())
            
    # ---- unfreeze after warmup epochs ----
    # if epoch + 1 == WARMUP_EPOCHS:
    #     for p in model.enc.parameters():
    #         p.requires_grad = True
    #     # rebuild optimizer & scheduler for the remaining steps
    #     optimizer = make_optim(unfrozen=True)
    #     remaining_steps = steps_per_epoch * (NUM_EPOCHS - (epoch + 1))
    #     warmup_rem = 0  # already warmed up; or set a small extra warmup if you like
    #     sched = get_linear_schedule_with_warmup(optimizer, warmup_rem, remaining_steps)

    # ---- validation ----
    model.eval()
    with torch.no_grad():
        batch_evaluation = model.evaluate_dataloader(eval_loader, id2label_ner, id2label_ee)
        history["ner_f1"].append(batch_evaluation["ner_f1"])
        history["ptr_acc"].append(batch_evaluation["ptr_acc"])
        history["ee_f1"].append(batch_evaluation["ee_f1"])
    print(f"ep{epoch+1}: NER F1={batch_evaluation['ner_f1']:.4f}  PTR@1={batch_evaluation['ptr_acc']:.4f}  EE mF1={batch_evaluation['ee_f1']:.4f}")

d:\GeoTKG\venv\Lib\site-packages\torch\optim\lr_scheduler.py:182: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


LOSS: 15.6114, STEP: 200
LOSS: 11.8092, STEP: 400
LOSS: 13.1526, STEP: 600
              precision    recall  f1-score   support

        DATE     0.7775    0.9166    0.8414      2459
    DURATION     0.4667    0.3750    0.4158       448
       EVENT     0.8643    0.8878    0.8759     16470
        TIME     0.1905    0.0571    0.0879        70

   micro avg     0.8440    0.8766    0.8600     19447
   macro avg     0.5747    0.5591    0.5552     19447
weighted avg     0.8417    0.8766    0.8581     19447

              precision    recall  f1-score   support

       AFTER     0.7646    0.6735    0.7162      7192
      BEFORE     0.6022    0.6912    0.6436      5790
    CONTAINS     0.7059    0.8155    0.7567      6948
      DURING     0.4737    0.0480    0.0872       375
      EQUALS     0.8019    0.1512    0.2545      1686
    IDENTITY     0.7866    0.9574    0.8636      3823
    OVERLAPS     0.3442    0.1485    0.2074       357

    accuracy                         0.7068     26171
  

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history["ner_loss"][::200], label="NER LOSS")
plt.plot(history["ee_loss"][::200], label="EE LOSS")
plt.plot(history["ca_loss"][::200], label="CA LOSS")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
history["ca_loss"][-1]